# Fig. S27 | Timing of pumping reduction

Compares pumping reductions during drought and recovery.

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

ROOT=next(p.resolve() for p in [Path.cwd(),*Path.cwd().parents] if (p/'src'/'management').is_dir())
DATA=ROOT/'outputs'/'MANAGEMENT_2012_2013'/'equal_volume'
OUT=ROOT/'outputs'/'figures'/'FigS27'/'FigS27.png'
OUT.parent.mkdir(parents=True,exist_ok=True)
data=pd.read_csv(DATA/'timing_seed_metrics.csv',dtype={'seed':str})
PI75=1.150349
colors={'2012 drought period':'#B77A61','2013 recovery period':'#486A9A'}
labels={'2012 drought period':'2012 drought','2013 recovery period':'2013 recovery'}
mpl.rcParams.update({'font.family':'Arial','font.size':11,'axes.labelsize':13,'axes.titlesize':12.5,'xtick.labelsize':10.5,'ytick.labelsize':10.5,'axes.linewidth':0.8,'axes.spines.top':True,'axes.spines.right':True})

In [ ]:
def summary(metric):
    rows=[]
    for (timing,budget),group in data.groupby(['timing','budget_nominal']):
        values=group[metric].to_numpy(float); mean=values.mean(); radius=PI75*values.std(ddof=0)
        rows.append((timing,budget,group.actual_dV_m3.mean(),mean,mean-radius,mean+radius))
    return pd.DataFrame(rows,columns=['timing','budget','volume','mean','low','high'])

fig,axes=plt.subplots(1,2,figsize=(7.0,3.2),sharey=True)
for ax,metric,title in [(axes[0],'eta_end','a  Intervention end'),(axes[1],'eta_6m','b  Six months later')]:
    stats=summary(metric)
    for timing in colors:
        d=stats[stats.timing.eq(timing)].sort_values('volume'); x=d.volume.to_numpy()/1e9
        ax.fill_between(x,d.low,d.high,color=colors[timing],alpha=0.14,lw=0)
        ax.plot(x,d['mean'],color=colors[timing],lw=2,label=labels[timing])
    ax.set(xlabel=r'Pumping reduction ($10^9$ m$^3$)',title=title,xlim=(0,5),ylim=(0.05,0.40))
    ax.set_xticks([0,1,2,3,4,5]); ax.set_yticks([0.1,0.2,0.3,0.4]); ax.grid(axis='y',color='#ECE9E4',lw=0.5)
    ax.title.set_fontweight('bold'); ax.title.set_ha('left'); ax.title.set_position((0,1))
axes[0].set_ylabel('Storage benefit per unit\npumping reduction')
axes[0].legend(frameon=False,fontsize=9,loc='lower right')
fig.tight_layout(w_pad=0.8); fig.savefig(OUT,dpi=600,bbox_inches='tight',facecolor='white'); plt.show()